In [ ]:



# https://unifiedportal-emp.epfindia.gov.in/publicPortal/no-auth/estSearch/searchEstablishment?_HDIV_STATE_=4-0-2475A9CF47C3CC6EEBAF2F6EF55AD44F

# _HDIV_STATE_=4-0-2475A9CF47C3CC6EEBAF2F6EF55AD44F
# {"EstName":"cogencis","EstCode":"","captcha":"AF8F4"}


# https://unifiedportal-emp.epfindia.gov.in/publicPortal/no-auth/estSearch/getDetails_3?_HDIV_STATE_=4-4-2475A9CF47C3CC6EEBAF2F6EF55AD44F
# _HDIV_STATE_=4-4-2475A9CF47C3CC6EEBAF2F6EF55AD44F
# {"EstId":"MHBAN0126734000"}

# LATER
# https://unifiedportal-emp.epfindia.gov.in/publicPortal/no-auth/estSearch/viewPaymentDetails/MHBAN0126734000?_HDIV_STATE_=4-17-2475A9CF47C3CC6EEBAF2F6EF55AD44F

# <div style="text-align: right;">        
# <a href="#" onclick="fnViewPaymentDtls('/publicPortal/no-auth/estSearch/viewPaymentDetails/MHBAN0126734000?_HDIV_STATE_=4-17-2475A9CF47C3CC6EEBAF2F6EF55AD44F');
#     return false;" title="Click to view payment details."><u>View Payment Details</u></a>
# <br>
# </div>

In [1]:
import requests
import json
from bs4 import BeautifulSoup
import pandas as pd
# import warnings
URL = "https://www.sebi.gov.in/sebiweb/other/OtherAction.do?doPmr=yes"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
    )
}

from utils import Helper

utils = Helper()
managers = utils.load_json(r"manager.json")

In [2]:
session = requests.Session()

resp = session.get(URL, headers=headers, verify=False)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

token_tag = soup.find("input", {"name": "org.apache.struts.taglib.html.TOKEN"})
if not token_tag:
    raise Exception("Token not found")

token = token_tag["value"]
payload = {
    "currdate": "",
    "loginflag": "0",
    "searchValue": "",
    "pmrId": "INP000008464@@INP000008464@@1729 ADVISORS LLP",
    "year": "2026",
    "month": "2",
    "format": "",
    "org.apache.struts.taglib.html.TOKEN": token,
    "loginEmail": "",
    "loginPassword": "",
    "cap_login": "",
    "moduleNo": "-1",
    "moduleId": "",
    "link": "",
    "yourName": "",
    "friendName": "",
    "friendEmail": "",
    "mailmessage": "",
    "cap_email": ""
}
post_headers = {
    **headers,
    "Referer": URL,
    "Origin": "https://www.sebi.gov.in"
}

year = {
    "2026": "2026",
    # "2025": "2025",
    # "2024": "2024",
    # "2023": "2023",
    # "2022": "2022",
    # "2021": "2021",
    # "2020": "2020",
    # "2019": "2019",
    # "2018": "2018"
}
month =  {
    # "1": "January",
    # "2": "February",
    # "3": "March",
    # "4": "April",
    # "5": "May",
    # "6": "June",
    # "7": "July",
    "8": "August",
    # "9": "September",
    # "10": "October",
    # "11": "November",
    # "12": "December"
}

manager = utils.load_json(r"manager.json")

C:\Users\rando\AppData\Roaming\Python\Python313\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.sebi.gov.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
import re
import time
import warnings
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")


def css(tag, prop):
    style = tag.get("style", "")
    m = re.search(rf"(?:^|;)\s*{re.escape(prop)}\s*:\s*([^;]+)", style, re.I)
    return m.group(1).strip() if m else None

def color(value):
    if not value:
        return None
    value = value.strip().lower().replace("#", "")
    if re.fullmatch(r"[0-9a-f]{6}", value):
        return value.upper()
    m = re.match(r"rgb\(\s*(\d+),\s*(\d+),\s*(\d+)\s*\)", value)
    return "".join(f"{int(x):02X}" for x in m.groups()) if m else None

def cell_text(cell):
    for br in cell.find_all("br"):
        br.replace_with("\n")
    return re.sub(r"[ \t]+", " ", cell.get_text(" ", strip=True))

def write_table(ws, table, start_row, company_name, yr, mon):
    rows = table.find_all("tr")
    occupied = set()
    thin = Side(style="thin", color="B7B7B7")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for r_idx, tr in enumerate(rows):
        row, col = start_row + r_idx, 5
        ws.cell(row, 1, company_name)
        ws.cell(row, 2, yr)
        ws.cell(row, 3, mon)

        for cell in tr.find_all(["td", "th"], recursive=False):
            while (row, col) in occupied:
                col += 1

            rowspan = max(1, int(cell.get("rowspan", 1) or 1))
            colspan = max(1, int(cell.get("colspan", 1) or 1))
            end_row = row + rowspan - 1
            end_col = col + colspan - 1

            for r in range(row, end_row + 1):
                for c in range(col, end_col + 1):
                    if (r, c) != (row, col):
                        occupied.add((r, c))

            x = ws.cell(row, col, cell_text(cell))
            x.font = Font(
                name="Arial", size=10,
                bold=cell.name == "th" or cell.find(["b", "strong"]) is not None,
                italic=cell.find(["i", "em"]) is not None
            )
            x.border = border

            align = css(cell, "text-align") or cell.get("align") or "left"
            x.alignment = Alignment(
                horizontal=align if align in ("left", "center", "right") else "left",
                vertical="center", wrap_text=True
            )

            bg = color(cell.get("bgcolor") or css(cell, "background-color"))
            if bg:
                x.fill = PatternFill(fill_type="solid", fgColor=bg)

            if rowspan > 1 or colspan > 1:
                ws.merge_cells(
                    start_row=row, start_column=col,
                    end_row=end_row, end_column=end_col
                )

            col = end_col + 1

    # Metadata is repeated even across HTML rowspan merges.
    for r in range(start_row, start_row + len(rows)):
        ws.cell(r, 1, company_name)
        ws.cell(r, 2, yr)
        ws.cell(r, 3, mon)

    return start_row + len(rows)

def write_empty(ws, start_row, company_name, yr, mon):
    for row in range(start_row, start_row + 10):
        ws.cell(row, 1, company_name)
        ws.cell(row, 2, yr)
        ws.cell(row, 3, mon)

        x = ws.cell(row, 5, "EMPTY")
        x.font = Font(color="FF0000", bold=True)
        x.fill = PatternFill(fill_type="solid", fgColor="FFEEEE")

    return start_row + 10

wb = Workbook()
ws = wb.active
ws.title = "SEBI_DATA"
ws.sheet_view.showGridLines = False

ws.column_dimensions["A"].width = 45
ws.column_dimensions["B"].width = 12
ws.column_dimensions["C"].width = 12
ws.column_dimensions["D"].width = 4

for col in range(5, 35):
    ws.column_dimensions[get_column_letter(col)].width = 20

current_row = 1
request_count = 0
token = None

for manager in managers[:1000]:
    company_name = manager["company_name"]
    pmr_id = manager["raw"]

    for yr in year:
        for mon, month_name in month.items():
            print(f"{company_name} | {yr} | {month_name}")

            if request_count % 10 == 0 or token is None:
                search_resp = session.post(URL, headers=post_headers, verify=False)
                search_resp.raise_for_status()

                soup = BeautifulSoup(search_resp.text, "html.parser")
                token_tag = soup.find("input", {"name": "org.apache.struts.taglib.html.TOKEN"})

                if not token_tag:
                    raise Exception("Token not found")

                token = token_tag["value"]

            payload = {
                "currdate": "",
                "loginflag": "0",
                "searchValue": "",
                "pmrId": pmr_id,
                "year": yr,
                "month": mon,
                "format": "",
                "org.apache.struts.taglib.html.TOKEN": token,
                "loginEmail": "",
                "loginPassword": "",
                "cap_login": "",
                "moduleNo": "-1",
                "moduleId": "",
                "link": "",
                "yourName": "",
                "friendName": "",
                "friendEmail": "",
                "mailmessage": "",
                "cap_email": ""
            }

            resp = session.post(URL, headers=post_headers, data=payload, verify=False)
            resp.raise_for_status()
            request_count += 1

            soup = BeautifulSoup(resp.text, "html.parser")
            tables = soup.find_all("table")
            tables = [t for t in tables if t.find_parent("table") is None]

            # Ignore tables without meaningful content.
            tables = [
                t for t in tables
                if t.find(["td", "th"]) and t.get_text(" ", strip=True)
            ]

            if not tables:
                print("    EMPTY")
                current_row = write_empty(ws, current_row, company_name, yr, month_name)
                continue

            for table in tables:
                current_row = write_table(
                    ws, table, current_row, company_name, yr, month_name
                )

            print(f"    {len(tables)} tables saved")
            time.sleep(.5)
    wb.save("SEBI_DATA.xlsx")

wb.save("SEBI_DATA.xlsx")
print("DONE!")